# Assignment 6: Applying AI-Based Style Transfer to an Image

## Objective
The objective of this assignment is to compare two different approaches to image style transfer:

1. **CycleGAN** using a pretrained Horse → Zebra model.
2. **Neural Style Transfer** using a VGG19-based approach and a painting as the style image.

The same original photograph is used for both methods. The final section places the Original, CycleGAN output, and Neural Style Transfer output side by side and provides a short comparison.

> **Note:** This notebook downloads the required pretrained models and sample images when it is run. An internet connection is required for the first run.

## 1. Install / Import Required Libraries
Run the installation cell if TensorFlow or TensorFlow Hub is not already installed.

In [ ]:
# If needed, uncomment and run this line in Google Colab/Jupyter:
# !pip install -q tensorflow tensorflow-hub tensorflow-datasets pillow matplotlib requests

import os
import io
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import requests
import tensorflow as tf
import tensorflow_hub as hub

print('TensorFlow version:', tf.__version__)
print('TensorFlow Hub imported successfully!')

## 2. Download the Images

For this demonstration, a horse photograph is used as the content image because the pretrained CycleGAN model can convert **horses to zebras**. A painting is used as the style image for Neural Style Transfer.

- **Content image:** Horse photograph
- **CycleGAN transformation:** Horse → Zebra
- **Style image:** Starry Night-style painting image


In [ ]:
import tensorflow_datasets as tfds

# Load one horse photograph from the official TensorFlow CycleGAN horse2zebra dataset.
# The same image is then used for both methods.
dataset, metadata = tfds.load(
    'cycle_gan/horse2zebra',
    with_info=True,
    as_supervised=True
)

horse_dataset = dataset['testA']
horse_image, _ = next(iter(horse_dataset.take(1)))
horse_image = tf.cast(horse_image, tf.uint8).numpy()
Image.fromarray(horse_image).save('content_image.jpg')

# TensorFlow's public Neural Style Transfer example image is used as the painting style.
STYLE_URL = 'https://storage.googleapis.com/download.tensorflow.org/example_images/style_transfer/style.jpg'

response = requests.get(STYLE_URL, timeout=30)
response.raise_for_status()
with open('style_image.jpg', 'wb') as f:
    f.write(response.content)

content_path = 'content_image.jpg'
style_path = 'style_image.jpg'

print('Horse content image saved as:', content_path)
print('Style image saved as:', style_path)


## 3. Load and Display the Original Images

Before applying any transformation, we inspect the content and style images.

In [ ]:
content_image = Image.open(content_path).convert('RGB')
style_image = Image.open(style_path).convert('RGB')

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(content_image)
plt.title('Original / Content Image')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(style_image)
plt.title('Style Image / Painting')
plt.axis('off')

plt.tight_layout()
plt.show()

## 4. Prepare the Content Image for CycleGAN

The pretrained CycleGAN model expects an image tensor. The image is resized while preserving a manageable resolution and converted to a floating-point tensor.

In [ ]:
def load_image_for_cyclegan(path, max_size=512):
    image = Image.open(path).convert('RGB')
    image.thumbnail((max_size, max_size))
    array = np.asarray(image).astype(np.float32)
    tensor = tf.convert_to_tensor(array)
    tensor = tensor / 127.5 - 1.0
    return tensor[tf.newaxis, ...]

content_tensor = load_image_for_cyclegan(content_path)
print('CycleGAN input shape:', content_tensor.shape)

## 5. Method 1 — CycleGAN

CycleGAN is a Generative Adversarial Network designed for **unpaired image-to-image translation**. It can learn a transformation between two visual domains without requiring every source image to have a directly corresponding target image.

### Cycle Consistency

A key idea is **cycle consistency**. If an image is translated from domain A to domain B and then translated back from B to A, the result should remain close to the original image. This encourages the model to change the desired visual domain while retaining important content.

Here we use a pretrained **Horse → Zebra** CycleGAN model.

In [ ]:
cyclegan_url = 'https://tfhub.dev/google/imagenet/cyclegan/horse2zebra/1'
cyclegan_model = hub.load(cyclegan_url)

print('Pretrained CycleGAN model loaded successfully!')

In [ ]:
cycle_output = cyclegan_model(content_tensor)

# TensorFlow Hub versions may return a tensor or a tuple/list containing a tensor.
if isinstance(cycle_output, (tuple, list)):
    cycle_output = cycle_output[0]
elif isinstance(cycle_output, dict):
    cycle_output = next(iter(cycle_output.values()))

cycle_output = tf.convert_to_tensor(cycle_output)
cycle_image = tf.clip_by_value((cycle_output[0] + 1.0) * 127.5, 0, 255)
cycle_image = tf.cast(cycle_image, tf.uint8).numpy()

print('CycleGAN output shape:', cycle_image.shape)

### CycleGAN Output

In [ ]:
plt.figure(figsize=(7, 5))
plt.imshow(cycle_image)
plt.title('CycleGAN Output — Horse to Zebra')
plt.axis('off')
plt.show()

## 6. Method 2 — Neural Style Transfer

Neural Style Transfer uses a pretrained CNN such as **VGG19** to separate an image into content and style representations. The generated image is optimized so that it retains important content features from the original image while matching the visual characteristics of the painting.

We use VGG19 feature layers to calculate:
- **Content loss:** difference between original and generated content features.
- **Style loss:** difference between style representations based on Gram matrices.

The generated image is then updated using gradient descent.

In [ ]:
def load_and_resize(path, max_dim=512):
    image = Image.open(path).convert('RGB')
    image.thumbnail((max_dim, max_dim))
    return np.array(image).astype(np.float32)

content_nst = load_and_resize(content_path)
style_nst = load_and_resize(style_path)

content_nst = content_nst[tf.newaxis, ...]
style_nst = style_nst[tf.newaxis, ...]

print('Content image shape:', content_nst.shape)
print('Style image shape:', style_nst.shape)

### Load VGG19

In [ ]:
vgg = tf.keras.applications.VGG19(include_top=False, weights='imagenet')
vgg.trainable = False

content_layer = 'block5_conv2'
style_layers = [
    'block1_conv1',
    'block2_conv1',
    'block3_conv1',
    'block4_conv1',
    'block5_conv1'
]

style_weight = 1e-2
content_weight = 1e4

print('VGG19 loaded successfully!')

### Define VGG Feature Extraction and Gram Matrix

In [ ]:
all_layers = style_layers + [content_layer]
outputs = [vgg.get_layer(name).output for name in all_layers]
feature_extractor = tf.keras.Model(vgg.input, outputs)

def gram_matrix(input_tensor):
    result = tf.linalg.einsum('bijc,bijd->bcd', input_tensor, input_tensor)
    input_shape = tf.shape(input_tensor)
    num_locations = tf.cast(input_shape[1] * input_shape[2], tf.float32)
    return result / num_locations

def preprocess_for_vgg(image):
    return tf.keras.applications.vgg19.preprocess_input(image)

def get_features(image):
    preprocessed = preprocess_for_vgg(image)
    outputs = feature_extractor(preprocessed)
    style_outputs = outputs[:len(style_layers)]
    content_outputs = outputs[len(style_layers):]
    style_outputs = [gram_matrix(style_output) for style_output in style_outputs]
    return style_outputs, content_outputs

In [ ]:
style_features, _ = get_features(style_nst)
_, content_features = get_features(content_nst)

print('Style and content features extracted successfully!')

### Optimize the Generated Image

In [ ]:
generated = tf.Variable(content_nst)
optimizer = tf.keras.optimizers.Adam(learning_rate=5.0)

def style_content_loss(outputs):
    style_outputs = outputs[:len(style_layers)]
    content_outputs = outputs[len(style_layers):]

    style_loss = tf.add_n([
        tf.reduce_mean((generated_style - target_style) ** 2)
        for generated_style, target_style in zip(style_outputs, style_features)
    ])
    style_loss *= style_weight / len(style_layers)

    content_loss = tf.add_n([
        tf.reduce_mean((generated_content - target_content) ** 2)
        for generated_content, target_content in zip(content_outputs, content_features)
    ])
    content_loss *= content_weight

    return style_loss + content_loss

@tf.function()
def train_step():
    with tf.GradientTape() as tape:
        outputs = feature_extractor(preprocess_for_vgg(generated))
        outputs = outputs[:len(style_layers)] + outputs[len(style_layers):]
        loss = style_content_loss(outputs)

    grad = tape.gradient(loss, generated)
    optimizer.apply_gradients([(grad, generated)])
    generated.assign(tf.clip_by_value(generated, 0.0, 255.0))
    return loss

In [ ]:
epochs = 10
steps_per_epoch = 10
loss_history = []

for epoch in range(epochs):
    for step in range(steps_per_epoch):
        loss = train_step()
    loss_history.append(float(loss.numpy()))
    print(f'Epoch {epoch + 1}/{epochs} - Loss: {loss.numpy():.2f}')

neural_style_output = tf.cast(tf.clip_by_value(generated[0], 0, 255), tf.uint8).numpy()
print('Neural Style Transfer completed!')

In [ ]:
plt.figure(figsize=(7, 5))
plt.imshow(neural_style_output)
plt.title('Neural Style Transfer Output')
plt.axis('off')
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(loss_history, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Style + Content Loss')
plt.title('Neural Style Transfer Loss')
plt.grid(True)
plt.show()

## 7. Final Side-by-Side Comparison
This is the main visual output required for the assignment.

In [ ]:
original_display = np.array(content_image)

plt.figure(figsize=(18, 6))

plt.subplot(1, 3, 1)
plt.imshow(original_display)
plt.title('1. Original Image')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(cycle_image)
plt.title('2. CycleGAN Output')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(neural_style_output)
plt.title('3. Neural Style Transfer Output')
plt.axis('off')

plt.tight_layout()
plt.show()

## 8. Save the Three Images and Comparison Figure

The generated files can be used directly when preparing the final submission document.

In [ ]:
Image.fromarray(original_display).save('01_original_image.png')
Image.fromarray(cycle_image).save('02_cyclegan_output.png')
Image.fromarray(neural_style_output).save('03_neural_style_output.png')

plt.figure(figsize=(18, 6))
plt.subplot(1, 3, 1)
plt.imshow(original_display)
plt.title('Original Image')
plt.axis('off')
plt.subplot(1, 3, 2)
plt.imshow(cycle_image)
plt.title('CycleGAN Output')
plt.axis('off')
plt.subplot(1, 3, 3)
plt.imshow(neural_style_output)
plt.title('Neural Style Transfer Output')
plt.axis('off')
plt.tight_layout()
plt.savefig('04_side_by_side_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

print('Saved: 01_original_image.png')
print('Saved: 02_cyclegan_output.png')
print('Saved: 03_neural_style_output.png')
print('Saved: 04_side_by_side_comparison.png')

## 9. Written Comparison

**Content preservation:** CycleGAN generally preserves the main structure and identity of the source object while translating it into the target visual domain. Neural Style Transfer also preserves the main content, but its texture and color changes can become more noticeable depending on the style image and optimization settings.

**Strength of style transformation:** Neural Style Transfer produces a more direct artistic transformation because the texture, colors, and visual patterns of the painting are intentionally transferred to the photograph. CycleGAN focuses on translating the image from one learned domain to another, such as horse to zebra.

**Difference between outputs:** The CycleGAN result changes domain-specific characteristics while keeping the photograph recognizable, whereas Neural Style Transfer overlays artistic textures and patterns while retaining the original scene structure. CycleGAN relies on **cycle consistency**, which encourages an image translated from one domain to another and back again to remain close to the original content.

### Conclusion
Both methods demonstrate different ideas behind AI-based image transformation. CycleGAN is useful for unpaired image-to-image domain translation, while VGG-based Neural Style Transfer is designed to combine the content of one image with the visual style of another.

## 10. Submission Checklist

- [x] Original image
- [x] CycleGAN output
- [x] Neural Style Transfer output
- [x] Side-by-side comparison
- [x] Short written comparison
- [x] Explanation of cycle consistency
- [x] Code and visual results included in the notebook